In [18]:
import os
os.chdir("/Users/mun-yeongmin/Downloads/") #working directory 설정

In [20]:
import pandas as pd
cargo25=pd.read_csv('경상남도_창원시_화물자동차운송사업자_20251218.csv', encoding="UTF-8")
cargo25.head()

,사업소명,시도명,시군구명,소재지도로명주소,소재지지번주소,전화번호,면허구분명,업종명,운영여부,차량대수,데이터기준일자
0,씨제이대한통운(주)창원지사,경상남도,창원시,"경상남도 창원시 마산합포구 해안대로 234 (신포동1가, (주)대한통운마산지점)",경상남도 창원시 마산합포구 신포동1가 42,000-0000-0000,운송사업,일반화물자동차운송사업,Y,254,2025-12-18
1,(주)엘엑스판토스,경상남도,창원시,경상남도 창원시 의창구 차상로 96 (팔용동),경상남도 창원시 의창구 팔용동 11-4,000-0000-0000,운송사업,일반화물자동차운송사업,Y,98,2025-12-18
2,한성특운(주),경상남도,창원시,경상남도 창원시 마산회원구 내서읍 중리공단로1길 31,경상남도 창원시 마산회원구 내서읍 중리 1127-4,000-0000-0000,운송사업,일반화물자동차운송사업,Y,93,2025-12-18
3,(주)공단특수육운,경상남도,창원시,경상남도 창원시 의창구 사화로 272 (주)공단특수육운 (팔용동),경상남도 창원시 의창구 팔용동 39-5 (주)공단특수육운,000-0000-0000,운송사업,일반화물자동차운송사업,Y,88,2025-12-18
4,세양상운(주),경상남도,창원시,경상남도 창원시 마산회원구 내서읍 중리공단로6길 29,경상남도 창원시 마산회원구 내서읍 중리 1118-2,000-0000-0000,운송사업,일반화물자동차운송사업,Y,80,2025-12-18


In [21]:
# 기본 확인
print(cargo25.shape)
print(cargo25.dtypes)
print(cargo25['운영여부'].value_counts())
print(cargo25['차량대수'].describe())

(170, 11)
사업소명        object
시도명         object
시군구명        object
소재지도로명주소    object
소재지지번주소     object
전화번호        object
면허구분명       object
업종명         object
운영여부        object
차량대수         int64
데이터기준일자     object
dtype: object
운영여부
Y    170
Name: count, dtype: int64
count    170.000000
mean      16.776471
std       27.445644
min        2.000000
25%        3.000000
50%        6.000000
75%       20.000000
max      254.000000
Name: 차량대수, dtype: float64


In [22]:
# 구 파싱
def extract_gu(address):
    gu_list = ['성산구', '의창구', '마산합포구', '마산회원구', '진해구']
    for gu in gu_list:
        if gu in str(address):
            return gu
    return '기타'

cargo25['구'] = cargo25['소재지도로명주소'].apply(extract_gu)
print(cargo25['구'].value_counts())

구
의창구      50
진해구      43
마산회원구    29
성산구      28
마산합포구    19
기타        1
Name: count, dtype: int64


In [23]:
# 구별 집계
gu_summary25 = cargo25.groupby('구')['차량대수'].agg(['sum','count','mean']).reset_index()
gu_summary25.columns = ['구', '총차량수', '업체수', '업체당평균차량']
gu_summary25['업체당평균차량'] = gu_summary25['업체당평균차량'].round(2)
print(gu_summary25.sort_values('총차량수', ascending=False))

       구  총차량수  업체수  업체당평균차량
2  마산회원구   841   29    29.00
4    의창구   636   50    12.72
5    진해구   530   43    12.33
1  마산합포구   413   19    21.74
3    성산구   399   28    14.25
0     기타    33    1    33.00


In [24]:
# 영세 비율
cargo25['영세여부'] = cargo25['차량대수'].apply(
    lambda x: '영세(5대이하)' if x <= 5 else '중대형(6대이상)'
)
영세 = cargo25.groupby(['구','영세여부']).size().unstack(fill_value=0)
영세['영세비율(%)'] = (영세['영세(5대이하)'] / 영세.sum(axis=1) * 100).round(1)
print(영세)

영세여부   영세(5대이하)  중대형(6대이상)  영세비율(%)
구                                  
기타            0          1      0.0
마산합포구        13          6     68.4
마산회원구         9         20     31.0
성산구          15         13     53.6
의창구          24         26     48.0
진해구          16         27     37.2


In [25]:
# 상위 5개 업체가 전체 차량의 몇 % 차지하는지
top5 = cargo25.nlargest(5, '차량대수')
print(top5[['사업소명', '구', '차량대수']])
print(f"상위 5개 비율: {top5['차량대수'].sum() / cargo25['차량대수'].sum() * 100:.1f}%")

             사업소명      구  차량대수
0  씨제이대한통운(주)창원지사  마산합포구   254
1       (주)엘엑스판토스    의창구    98
2         한성특운(주)  마산회원구    93
3       (주)공단특수육운    의창구    88
4         세양상운(주)  마산회원구    80
상위 5개 비율: 21.5%


In [26]:
# 진해구 업체만 뽑아서 규모 분포 확인
jinhae25 = cargo25[cargo25['구'] == '진해구']
print(jinhae25[['사업소명', '차량대수']].sort_values('차량대수', ascending=False))

                사업소명  차량대수
17       (유)해강물류보세창고    48
18        ㈜올웨이즈익스프레스    47
19         (주)월드로지스텍    44
24         디앤디로직스(주)    32
28           선도특운(주)    27
30            (주)더트럭    24
31           (주)강진상운    24
38           금륜물류(주)    23
43        (주)보고대성로지스    20
46         지알로지스틱(주)    17
45   씨제이대한통운(주)진해영업소    17
49          퍼팩트특송(주)    15
52           조은물류(주)    14
55        주식회사 디지플래너    13
62        (주)신항특수로직스    11
64        제이에스모터스(주)    11
65        허브로지스틱스(주)    10
66             개인사업자    10
68           (주)신우상운    10
69           (주)다온통운     9
72        (주)태광익스프레스     9
73          (주)길한로직스     8
77       (주)포스텍 죽곡지점     7
78           (주)제일급수     7
80        신지로지스틱스(주)     7
82           (주)나우물류     6
89           (주)후남운수     6
94           (주)이지경남     5
117          진해렉카(주)     4
116           진해도완렉카     4
114           조은정비렉카     4
108          (주)창원특송     4
103         (주)금성로직스     4
102        (주)성산지엘에스     4
100        씨엠케이라인(주)     4
99       주식회사 제이제이물류     4
1